In [2]:
import scipy.stats as stats
import pandas as pd

In [3]:
df = pd.read_csv('goldman_sachs.csv')

In [4]:
account_stats = df.groupby('AccountID').agg(
    TxnCount=('TransactionAmount', 'count'),
    AvgBalance=('AccountBalance', 'mean')
).reset_index()

In [5]:
median_txn = account_stats['TxnCount'].median()

high_volume = account_stats[
    account_stats['TxnCount'] > median_txn
]['AvgBalance']

low_volume = account_stats[
    account_stats['TxnCount'] <= median_txn
]['AvgBalance']

In [6]:
t_stat, p_value = stats.ttest_ind(
    high_volume,
    low_volume,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 0.31506769364079873
P-value: 0.7530534149746722


In [7]:
if p_value < 0.05:
    print("Reject H0: High-volume accounts have significantly higher average balances.")
else:
    print("Fail to reject H0: No significant difference in average balances.")

Fail to reject H0: No significant difference in average balances.


In [9]:
txn_count = df.groupby('AccountID').size().reset_index(name='TxnCount')

def activity_level(x):
    if x > 20:
        return 'High'
    elif x >= 10:
        return 'Medium'
    else:
        return 'Low'

txn_count['ActivityLevel'] = txn_count['TxnCount'].apply(activity_level)


avg_balance = df.groupby('AccountID')['AccountBalance'].mean().reset_index()

segmented_data = txn_count.merge(avg_balance, on='AccountID')


In [11]:
high = segmented_data[segmented_data['ActivityLevel'] == 'High']['AccountBalance']
medium = segmented_data[segmented_data['ActivityLevel'] == 'Medium']['AccountBalance']
low = segmented_data[segmented_data['ActivityLevel'] == 'Low']['AccountBalance']


In [12]:
f_stat, p_value = stats.f_oneway(high, medium, low)

print("F-statistic:", f_stat)
print("P-value:", p_value)


F-statistic: nan
P-value: nan


C:\Users\DELL\anaconda3\Lib\site-packages\scipy\stats\_stats_py.py:4133: DegenerateDataWarning: at least one input has length 0
  warnings.warn(stats.DegenerateDataWarning('at least one input '


In [13]:
if p_value < 0.05:
    print("Reject H0: Average balances differ across activity segments.")
else:
    print("Fail to reject H0: No significant difference across segments.")


Fail to reject H0: No significant difference across segments.
